In [11]:
from dotenv import load_dotenv
import os

load_dotenv()

aws_access_key = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")
aws_region = os.getenv("AWS_DEFAULT_REGION")

print("Credentials loaded successfully")



python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 13


Credentials loaded successfully


In [ ]:
file_path = "s3://aws-ai-cost-analysis/raw/aws_dummy_billing_data.csv"

: 

: 

In [ ]:
import pandas as pd
df = pd.read_csv(file_path)

: 

In [ ]:
df.head(5)

,UsageDate,Service,Region,UnblendedCost,UsageType,Account
0,01-02-2026,AmazonEC2,us-east-1,45.20,BoxUsage:t3.medium,prod
1,01-02-2026,AmazonS3,us-west-2,12.05,TimedStorage-ByteHrs,prod
2,02-02-2026,AmazonRDS,us-east-1,85.00,InstanceUsage:db.t3.micro,QA
3,02-02-2026,AmazonEC2,us-east-1,42.10,BoxUsage:t3.medium,dev
4,03-02-2026,AmazonS3,us-west-2,11.50,TimedStorage-ByteHrs,dev


: 

In [14]:
df.info
df.columns

Index(['UsageDate', 'Service', 'Region', 'UnblendedCost', 'UsageType',
       'Account'],
      dtype='str')

In [15]:
# 1. Rename 'UsageDate' to 'Usage'
df = df.rename(columns={'UsageDate': 'Usage'})

# 2. Convert the newly named 'Usage' column to datetime
df['Usage'] = pd.to_datetime(df['Usage'], dayfirst=True)

# 3. Clean the cost column
df['UnblendedCost'] = pd.to_numeric(df['UnblendedCost'], errors='coerce')

# 4. Drop rows with missing costs
df = df.dropna(subset=['UnblendedCost'])

# 5. Check results
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Usage          20 non-null     datetime64[us]
 1   Service        20 non-null     str           
 2   Region         20 non-null     str           
 3   UnblendedCost  20 non-null     float64       
 4   UsageType      20 non-null     str           
 5   Account        20 non-null     str           
dtypes: datetime64[us](1), float64(1), str(4)
memory usage: 1.9 KB


In [16]:
df.head(3)

,Usage,Service,Region,UnblendedCost,UsageType,Account
0,2026-02-01,AmazonEC2,us-east-1,45.20,BoxUsage:t3.medium,prod
1,2026-02-01,AmazonS3,us-west-2,12.05,TimedStorage-ByteHrs,prod
2,2026-02-02,AmazonRDS,us-east-1,85.00,InstanceUsage:db.t3.micro,QA


In [18]:
df["Year"] = df["Usage"].dt.year
df["Month"] = df["Usage"].dt.month
df["Day"] = df["Usage"].dt.day
df["MonthName"] = df["Usage"].dt.strftime("%B")


In [19]:
import pandas as pd

# Convert date
df["Usage"] = pd.to_datetime(df["Usage"], dayfirst=True, errors="coerce")

# Convert numeric columns

df["UnblendedCost"] = pd.to_numeric(df["UnblendedCost"], errors="coerce")

# Remove invalid records
df = df.dropna(subset=["Usage", "UnblendedCost"])
df = df[df["UnblendedCost"] >= 0]

# Remove duplicates
df = df.drop_duplicates()

df.head()


,Usage,Service,Region,UnblendedCost,UsageType,Account,Year,Month,Day,MonthName
0,2026-02-01,AmazonEC2,us-east-1,45.20,BoxUsage:t3.medium,prod,2026,2,1,February
1,2026-02-01,AmazonS3,us-west-2,12.05,TimedStorage-ByteHrs,prod,2026,2,1,February
2,2026-02-02,AmazonRDS,us-east-1,85.00,InstanceUsage:db.t3.micro,QA,2026,2,2,February
3,2026-02-02,AmazonEC2,us-east-1,42.10,BoxUsage:t3.medium,dev,2026,2,2,February
4,2026-02-03,AmazonS3,us-west-2,11.50,TimedStorage-ByteHrs,dev,2026,2,3,February


In [20]:
import pandas as pd

# Assuming your 'df' is already cleaned using the code we discussed earlier

# --- 1. Cost by Service ---
# Groups by service, sums the cost, and sorts from highest spender to lowest
cost_by_service = (
    df.groupby("Service")["UnblendedCost"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

# --- 2. Cost by Region ---
cost_by_region = (
    df.groupby("Region")["UnblendedCost"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

# --- 3. Cost by Account ---
cost_by_account = (
    df.groupby("Account")["UnblendedCost"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

# --- Printing the Results ---
print("🔥 TOP SERVICES BY SPEND:")
print(cost_by_service.head(10)) 

print("\n🌍 SPEND BY REGION:")
print(cost_by_region)

print("\n💰 SPEND BY ACCOUNT:")
print(cost_by_account)

🔥 TOP SERVICES BY SPEND:
            Service  UnblendedCost
0         AmazonRDS         340.00
1         AmazonEC2         283.25
2          AmazonS3          53.75
3    AmazonDynamoDB          22.40
4           AWSGlue           8.20
5  AmazonCloudFront           5.60
6     AmazonRoute53           2.00
7         AWSLambda           0.45

🌍 SPEND BY REGION:
         Region  UnblendedCost
0     us-east-1         594.25
1  eu-central-1          60.05
2     us-west-2          53.75
3        global           7.60

💰 SPEND BY ACCOUNT:
     Account  UnblendedCost
0       prod         240.20
1  analytics         216.00
2        dev         144.65
3         QA         114.80


In [ ]:
# Simple Anomaly Detection using Standard Deviation
mean_cost = df['UnblendedCost'].mean()
std_cost = df['UnblendedCost'].std()

# Define a threshold (e.g., 3 times the standard deviation)
threshold = 3
df['is_anomaly'] = df['UnblendedCost'] > (mean_cost + (threshold * std_cost))

# View only the anomalies
anomalies = df[df['is_anomaly'] == True]
print(anomalies)

Empty DataFrame
Columns: [Usage, Service, Region, UnblendedCost, UsageType, Account, Year, Month, Day, MonthName, is_anomaly]
Index: []


: 

: 

In [ ]:
# Create the summary dictionary
summary = {
    "Total Cost": df["UnblendedCost"].sum(),
    
    # .get() or checking empty ensures the code doesn't break if data is missing
    "Top Service": cost_by_service.iloc[0]["Service"] if not cost_by_service.empty else "N/A",
    
    "Top Region": cost_by_region.iloc[0]["Region"] if not cost_by_region.empty else "N/A",
    
    # Updated to 'Account' based on our previous CSV cleaning
    "Top Account": cost_by_account.iloc[0]["Account"] if not cost_by_account.empty else "N/A",
    
    "Anomaly Count": len(anomalies) if 'anomalies' in locals() else 0
}

# Print the result nicely
import json
print(json.dumps(summary, indent=4, default=str))

{
    "Total Cost": 715.6500000000001,
    "Top Service": "AmazonRDS",
    "Top Region": "us-east-1",
    "Top Account": "prod",
    "Anomaly Count": 0
}


: 

: 

In [21]:
# 1. Ensure the column is in datetime format
df["Usage"] = pd.to_datetime(df["Usage"])

# 2. Create a 'Month' column (Extract Year and Month)
# .dt.to_period('M') turns '2026-02-15' into '2026-02'
df["Month"] = df["Usage"].dt.to_period("M")

# Group by both the period (for sorting) and the name (for display)
monthly_spend = df.groupby(["Month", "MonthName"])["UnblendedCost"].sum().reset_index()

# Sort by 'Month' to ensure they appear in chronological order, not alphabetical
monthly_spend = monthly_spend.sort_values("Month")

print(monthly_spend)

     Month MonthName  UnblendedCost
0  2026-02  February         715.65


In [ ]:
df.head()

,UsageDate,Service,Region,UnblendedCost,UsageType,Account
0,01-02-2026,AmazonEC2,us-east-1,45.20,BoxUsage:t3.medium,prod
1,01-02-2026,AmazonS3,us-west-2,12.05,TimedStorage-ByteHrs,prod
2,02-02-2026,AmazonRDS,us-east-1,85.00,InstanceUsage:db.t3.micro,QA
3,02-02-2026,AmazonEC2,us-east-1,42.10,BoxUsage:t3.medium,dev
4,03-02-2026,AmazonS3,us-west-2,11.50,TimedStorage-ByteHrs,dev


: 

SAVE THE FILE TO s3 BUCKET AS CLEAN DATASET

In [36]:
# # ** (use the file only to save the data to S3) **

from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

processed_path = f"s3://aws-ai-cost-analysis/processed/cleaned_aws_dummy_cost_data{timestamp}.csv"
df.to_csv(
    processed_path,
    index=False,
    storage_options={
        "key": aws_access_key,
        "secret": aws_secret_key,
        "client_kwargs": {"region_name": aws_region}
    }
)

print("Versioned file saved.")

Versioned file saved.


Connect to RDS and upload the clean dataset to RDS for further analysis.

S3 → Read File → Pandas DataFrame → PostgreSQL RDS

🔹 1️⃣ EXTRACT (From S3 → Pandas)
obj = s3.get_object(Bucket=bucket_name, Key=file_key)

df = pd.read_csv(io.BytesIO(obj["Body"].read()))

That’s your E (Extract) step.

🔹 2️⃣ LOAD (Pandas → PostgreSQL RDS)
df.to_sql(
    table_name,
    engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000
)

That’s your L (Load) step.

🧠 Where is the "T" (Transform)?

The Transform step happens inside Pandas before .to_sql().

Example:

# Example transformations
df.drop_duplicates(inplace=True)
df["created_at"] = pd.to_datetime(df["created_at"])
df = df[df["amount"] > 0]

That’s your T (Transform).

🎯 So the Minimal Production Flow Is:
# EXTRACT
obj = s3.get_object(Bucket=bucket_name, Key=file_key)
df = pd.read_csv(io.BytesIO(obj["Body"].read()))

# TRANSFORM (if needed)
# df = ...

# LOAD
df.to_sql(table_name, engine, if_exists="append", index=False)

In [14]:
#---------------
# Connect to RDS
#------------------------

from dotenv import load_dotenv
import os
from sqlalchemy import create_engine

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DUMMY_DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DUMMY_DB_NAME')}"
)

print("Connected ✅")

python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 16


Connected ✅


In [8]:
import boto3
import pandas as pd
import io
import os
from dotenv import load_dotenv
from pathlib import Path


# Load environment variables
load_dotenv()

# -----------------------------
# Step 1: Read Environment Vars
# -----------------------------
bucket_name = os.getenv("S3_BUCKET")
file_key = os.getenv("DUMMY_S3_PROCESSED_KEY")

# Debug check
if not bucket_name or not file_key:
    raise ValueError("❌ S3 bucket or file key not found in environment variables")

# -----------------------------
# Step 2: Connect to S3
# -----------------------------
s3 = boto3.client('s3')

obj = s3.get_object(
    Bucket=bucket_name,
    Key=file_key
)

print("Connected to S3 ✅")


python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 13


Connected to S3 ✅


In [9]:
# Read file from S3
file_content = obj['Body'].read()

# Convert to DataFrame
df = pd.read_csv(io.BytesIO(file_content))

print("Loaded into DataFrame ✅")
print(df.head())

Loaded into DataFrame ✅
        Usage    Service     Region  UnblendedCost                  UsageType  \
0  2026-02-01  AmazonEC2  us-east-1          45.20         BoxUsage:t3.medium   
1  2026-02-01   AmazonS3  us-west-2          12.05       TimedStorage-ByteHrs   
2  2026-02-02  AmazonRDS  us-east-1          85.00  InstanceUsage:db.t3.micro   
3  2026-02-02  AmazonEC2  us-east-1          42.10         BoxUsage:t3.medium   
4  2026-02-03   AmazonS3  us-west-2          11.50       TimedStorage-ByteHrs   

  Account  Year    Month  Day MonthName  
0    prod  2026  2026-02    1  February  
1    prod  2026  2026-02    1  February  
2      QA  2026  2026-02    2  February  
3     dev  2026  2026-02    2  February  
4     dev  2026  2026-02    3  February  


In [15]:
load_dotenv()

# -----------------------------
# Step 1: Read Environment Vars
# -----------------------------
table_name = os.getenv("DUMMY_DB_NAME")


df.to_sql(
    table_name,
    engine,
    if_exists="append",   # use "replace" if you want to overwrite
    index=False,
    method="multi"
)

print("Data loaded to RDS successfully 🚀")

python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 16


Data loaded to RDS successfully 🚀


In [18]:
from sqlalchemy import text

with engine.connect() as connection:
    result = connection.execute(text("SELECT current_database();"))
    print("Connected to:", result.scalar())

Connected to: aws_dummy_cost_data


In [19]:
# -----------------------------
# Step 5: Verify Table
# -----------------------------
with engine.connect() as connection:
    result = connection.execute(text(f"""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema='public';
    """))
    tables = [row[0] for row in result]
    print("Tables in DB:", tables)
    
    # Optional: see first 5 rows
    sample = connection.execute(text(f"SELECT * FROM {table_name} LIMIT 5;"))
    for row in sample:
        print(row)

Tables in DB: ['aws_dummy_cost_data']
('2026-02-01', 'AmazonEC2', 'us-east-1', 45.2, 'BoxUsage:t3.medium', 'prod', 2026, '2026-02', 1, 'February')
('2026-02-01', 'AmazonS3', 'us-west-2', 12.05, 'TimedStorage-ByteHrs', 'prod', 2026, '2026-02', 1, 'February')
('2026-02-02', 'AmazonRDS', 'us-east-1', 85.0, 'InstanceUsage:db.t3.micro', 'QA', 2026, '2026-02', 2, 'February')
('2026-02-02', 'AmazonEC2', 'us-east-1', 42.1, 'BoxUsage:t3.medium', 'dev', 2026, '2026-02', 2, 'February')
('2026-02-03', 'AmazonS3', 'us-west-2', 11.5, 'TimedStorage-ByteHrs', 'dev', 2026, '2026-02', 3, 'February')
